# Module 1 Assignment Project

Designing a simple data product (dashboard) using real Singapore Job's Posting csv data. 

Our team is addressing a gap faced by companies entering the Singapore market: no reliable, industry-specific benchmark for talent costs or local hiring pool depth, even though headcount is typically the largest line item in an entry budget.

Our goal is to establish insights for clients, so that they can select their target industry and receive an evidence-based view of prevailing salary ranges by industry or by job title roles with data from 2023 - 2024.

### Structure

- Load and inspect initial EDA
- Remove missing values
- Clean date formats
- Clean title and text
- Remove duplicate postings
- Validate positions levels and experience
- Clean and flag unreliable salaries
- Convert hourly and annual salaries to monthly
- Parse the categories JSON
- .explode() into the long category table
- Job roles filtering from 'title_clean'
- Build analysis groupings
- Write the cleaned files
- Build the Streamlit dashboard

> The one design decision that matters is to acknowledge categories is one-to-many: a posting can sit in several industries. We keep a wide table at one row per posting for all headline numbers, and a separate exploded table at one row per posting x category for industry breakdowns.

In [1]:
import pandas as pd
import numpy as np
import re
import json

## 1. Exploratory Data Analysis

In [3]:
df = pd.read_csv('SGJobData.csv')

#Initial exploratory info

df.info()
df.describe(include='all')


<class 'pandas.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  str    
 1   employmentTypes                     1044597 non-null  str    
 2   metadata_expiryDate                 1044597 non-null  str    
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  str    
 5   metadata_newPostingDate             1044597 non-null  str    
 6   metadata_originalPostingDate        1044597 non-null  str    
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVacancies     

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
count,1044597,1044597,1044597,1048585,1044597,1044597,1044597,1.048585e+06,1.048585e+06,1.048585e+06,...,0.0,1044597,1044597,1.048585e+06,1.048585e+06,1044597,1048585.0,1044597,1044597,1.048585e+06
unique,21125,8,453,2,1044597,431,603,NaN,NaN,NaN,...,NaN,9,53151,NaN,NaN,1,NaN,3,377084,NaN
top,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-07-28,False,MCF-2023-0252866,2023-06-09,2023-07-14,NaN,NaN,NaN,...,NaN,Executive,THE SUPREME HR ADVISORY PTE. LTD.,NaN,NaN,Monthly,NaN,Open,SUPERVISOR,NaN
freq,92869,458139,4487,986717,1,4508,4029,NaN,NaN,NaN,...,NaN,253701,61638,NaN,NaN,1044597,NaN,902614,8331,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.472327e-02,2.136571e+00,2.674536e+01,...,NaN,NaN,NaN,5.723578e+03,3.815312e+03,NaN,0.0,NaN,NaN,4.769445e+03
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.822675e-01,1.062612e+01,8.262001e+01,...,NaN,NaN,NaN,5.018387e+04,3.172182e+03,NaN,0.0,NaN,NaN,2.547809e+04
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,...,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,0.0,NaN,NaN,0.000000e+00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,1.000000e+00,...,NaN,NaN,NaN,3.300000e+03,2.500000e+03,NaN,0.0,NaN,NaN,2.900000e+03
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,4.000000e+00,...,NaN,NaN,NaN,4.500000e+03,3.000000e+03,NaN,0.0,NaN,NaN,3.800000e+03
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,1.000000e+00,1.700000e+01,...,NaN,NaN,NaN,6.500000e+03,4.500000e+03,NaN,0.0,NaN,NaN,5.500000e+03


### Initial information gathered

- Total of **1048585 rows** and **22 columns**
- Some have missing rows (showing 1044597 only): **3988 missing values**
- Column **'occupationID'** is totally empty (all Null values)
- **Dated columns listed as object** --> Indicating string format (need to convert)
- Column **'categories' listed as object** (looks like json, but in string format; to check)
- 'salary_minimum' and 'salary_maximum' have **outliers**

## 1.1 Removing Missing Values (NaN)

In [4]:
# Checking missing values

missing_values = df.isna().sum()

print(missing_values[missing_values > 0].sort_values(ascending=False))

occupationId                    1048585
categories                         3988
employmentTypes                    3988
metadata_expiryDate                3988
metadata_jobPostId                 3988
metadata_newPostingDate            3988
metadata_originalPostingDate       3988
positionLevels                     3988
postedCompany_name                 3988
salary_type                        3988
status_jobStatus                   3988
title                              3988
dtype: int64


Similar number of missing values. To drop these rows as it does not help in analysis (no usable data value).

In [5]:
# Entire row of occupationId is empty. Initial df.dropna() via rows will drop every single row. Hence drop via column.

# Drop empty occupationId column
df_clean = df.drop(columns=['occupationId'])

# We use jobs id column which is unique, except for the 3988 empty rows, to drop the empty jobs id rows.
df_clean = df_clean.dropna(subset=['metadata_jobPostId']).copy()

#check if rows are dropped
df_clean.isna().sum()

categories                            0
employmentTypes                       0
metadata_expiryDate                   0
metadata_isPostedOnBehalf             0
metadata_jobPostId                    0
metadata_newPostingDate               0
metadata_originalPostingDate          0
metadata_repostCount                  0
metadata_totalNumberJobApplication    0
metadata_totalNumberOfView            0
minimumYearsExperience                0
numberOfVacancies                     0
positionLevels                        0
postedCompany_name                    0
salary_maximum                        0
salary_minimum                        0
salary_type                           0
status_id                             0
status_jobStatus                      0
title                                 0
average_salary                        0
dtype: int64

## 1.2 Parsing Date type

In [6]:
# Parsing the date columns - convert date object to datetime 

date_columns = ['metadata_expiryDate', 'metadata_newPostingDate', 'metadata_originalPostingDate']

for c in date_columns:
    df_clean[c] = pd.to_datetime(df_clean[c], errors='coerce')

In [7]:
print('Check for unparseable dates:', {c: df_clean[c].isna().sum() for c in date_columns})
print('Data date range:', df_clean['metadata_originalPostingDate'].min().date(), '->',
                     df_clean['metadata_originalPostingDate'].max().date())

Check for unparseable dates: {'metadata_expiryDate': np.int64(0), 'metadata_newPostingDate': np.int64(0), 'metadata_originalPostingDate': np.int64(0)}
Data date range: 2022-10-03 -> 2024-05-29


## 1.3 Cleaning 'title' Column

'title' column was extremely messy with various forms of input since it is free text. To avoid over-normalizing the data,
Removing the following:
- Whitespaces before and after
- hashtags
- Punctuations and emojis
- Recruiter Codes at the start of entry
- Any other recruiter codes within the text line or salary

In [8]:
REGEX_RULES = [
    ('recruiter_code', r'^\s*\d{3,6}\s*[-:]\s*', ' '),          # recruiter codes at the start
    ('hashtags', r'#\w+', ' '),                                 # hashtags
    ('other_numbers', r'\b(?=[a-z0-9]*\d)[a-z0-9]{3,}\b', ' '), # any other numbers of 3 and above
    ('punction_emojis', r'[^a-z]', ' '),                        # punctuations, emojis
    ('collapse_ws', r'\s+', ' '),                               # collapse whitespace
]

lower_title = df_clean['title'].str.lower()                     # make all cell values lowercase

for name, pattern, replace in REGEX_RULES:                      # reassignment with regex, replace if pattern matches
    lower_title = lower_title.str.replace(pattern, replace, regex=True)

df_clean['title_clean'] = lower_title.str.strip()               # keep 'title' column, create new column 'title_clean' with normalized entry


# after cleaning, some rows become empty. if so, replace with original 'title'

df_clean['title_clean'] = df_clean['title_clean'].where(
    df_clean['title_clean'] != '', df_clean['title'].str.lower())

## 1.4 Removing duplicated postings

Removing any duplicated postings due to original postings expiring. 4 parameters elected to identify duplicate rows:
- Using cleaned titles: 'title_clean'
- Using company name: 'postedCompany_name'
- Using salary: 'salary_minimum' and 'salary_maximum'

In [9]:
# Duplicated postings, defined as new postings of similar job due to previous posting expiring.

key_parameters = ['title_clean', 'postedCompany_name', 'salary_minimum', 'salary_maximum']

# Sort by original posting date first, drop duplicates but keep first posting
df_clean = df_clean.sort_values('metadata_originalPostingDate').drop_duplicates(subset= key_parameters, keep='first')

print(df_clean.shape)


(629246, 22)


## 1.5 Validating Position Levels against Years of Experience

To clean for positions with contradicting position levels vs years of experience

In [10]:
# Contradictions between stated job level and required experience.
JUNIOR_LEVELS = ['Fresh/entry level', 'Junior Executive']
SENIOR_LEVELS = ['Middle Management', 'Senior Management', 'Senior Executive']

years = df_clean['minimumYearsExperience']

# Senior label but almost no experience required
too_junior_for_label = df_clean['positionLevels'].isin(SENIOR_LEVELS) & (years < 3)

# Junior label but decades of experience required
too_senior_for_label = df_clean['positionLevels'].isin(JUNIOR_LEVELS) & (years > 20)

# Implausible experience regardless of label
implausible_years = years > 40

contradictory = too_junior_for_label | too_senior_for_label

print(f'Senior label, under 3 years:  {too_junior_for_label.sum():,}')
print(f'Junior label, over 20 years:  {too_senior_for_label.sum():,}')
print(f'Experience over 40 years:     {implausible_years.sum():,}')
print(f'Total contradictory labels:   {contradictory.sum():,} '
      f'({contradictory.mean():.2%} of postings)')

# Keep the row, blank the untrustworthy value. If we drop rows, it will affect hiring pool depth.
df_clean['positionLevels_clean'] = df_clean['positionLevels'].where(~contradictory)
df_clean.loc[implausible_years, 'minimumYearsExperience'] = np.nan

print(f"\npositionLevels_clean is null for "
      f"{df_clean['positionLevels_clean'].isna().sum():,} postings "
      f"({df_clean['positionLevels_clean'].isna().mean():.1%})")
df_clean['positionLevels_clean'].value_counts(dropna=False)

Senior label, under 3 years:  16,408
Junior label, over 20 years:  28
Experience over 40 years:     19
Total contradictory labels:   16,436 (2.61% of postings)

positionLevels_clean is null for 16,436 postings (2.6%)


positionLevels_clean
Executive            152290
Junior Executive      94169
Non-executive         81278
Professional          73458
Manager               72111
Fresh/entry level     58517
Senior Executive      48582
NaN                   16436
Middle Management     16372
Senior Management     16033
Name: count, dtype: int64

## 1.6 Dealing with Salary Outliers

Setting a boundary for the monthly salary to avoid outliers, not considering the typos for hourly rates and annual rates.

In [11]:
sal = ['salary_minimum', 'salary_maximum']

SALARY_FLOOR, SALARY_CEILING = 500, 60000

# Replace any empty salary with NaN
df_clean[sal] = df_clean[sal].replace(0, np.nan)

# Rebuild without 0 salary job postings
df_clean['average_salary'] = df_clean[sal].mean(axis=1)

# Set monthly pay range to remove any typos or wrong entries
df_clean_salary = df_clean[df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING)].copy()

## 1.6.1 Converting hourly and annual salaries to Monthly
We want to take into account typos for hour and annual salaries to be viable data to keep integrity of the data.

#### Rules:

| Rule | Reading | Action |
|---|---|---|
| `salary_min_clean` < 100 | hourly rate | x hours x 52 / 12 (44.3h full-time, 21h part-time, MoM averages) |
| min < 0.2 x max, max > 20,500 **AND** max/12 >= min | max quoted annually | max / 12 |
| min < 0.2 x max, max <= 20,500 | range too wide to trust | both set to the raw average |
| min > 40,000 | both quoted annually | both / 12 |



In [13]:
#####SETUP####
# Work from the raw bounds; only rows with a present value get touched.
df_clean['salary_min_clean'] = df_clean['salary_minimum']
df_clean['salary_max_clean'] = df_clean['salary_maximum']

has_min = df_clean['salary_min_clean'].notna()
has_max = df_clean['salary_max_clean'].notna()
both = has_min & has_max

# For rows with unreliable small maximum or extremely high maximum: unusable, blank it rather than drop the posting
df_clean.loc[has_max & ((df_clean['salary_max_clean'] < 5) | (df_clean['salary_max_clean'] >= 1000000)),
             ['salary_min_clean', 'salary_max_clean']] = np.nan
has_min = df_clean['salary_min_clean'].notna()
has_max = df_clean['salary_max_clean'].notna()
both = has_min & has_max

# For rows if only the maximum was filled in properly: mirror it into the minimum
only_max_ok = both & (df_clean['salary_min_clean'] < 1000) & \
              (df_clean['salary_min_clean'] < 0.2 * df_clean['salary_max_clean'])
df_clean.loc[only_max_ok, 'salary_min_clean'] = df_clean.loc[only_max_ok, 'salary_max_clean']
print(f'Minimum mirrored from maximum: {only_max_ok.sum():,}')

Minimum mirrored from maximum: 1,330


In [14]:
# Changing potential hourly wages to monthly
# Monthly = hourly x weekly hours x 52 / 12.  
# MoM average hours: 44.3 full-time, 21 part-time. https://stats.mom.gov.sg

hourly = df_clean['salary_min_clean'].notna() & (df_clean['salary_min_clean'] < 100)

# A senior role quoted at an hourly-looking rate is a data error, not an
# hourly job. Flag rather than convert.
salary_level_mismatch = hourly & df_clean['positionLevels_clean'].isin(SENIOR_LEVELS)
hourly = hourly & ~salary_level_mismatch

part_time = df_clean['employmentTypes'].eq('Part Time')
hours = np.where(part_time, 21, 44.3)
factor = pd.Series(hours * 52 / 12, index=df_clean.index)

for column in ['salary_min_clean', 'salary_max_clean']:
    df_clean.loc[hourly, column] = df_clean.loc[hourly, column] * factor[hourly]

print(f'Converted from hourly:        {hourly.sum():,}')
print(f'Senior roles at hourly rates: {salary_level_mismatch.sum():,} (flagged, not converted)')

Converted from hourly:        3,361
Senior roles at hourly rates: 39 (flagged, not converted)


In [15]:
# Changing potential annual wages to monthly

# A maximum is read as annual only if it is too large to be monthly AND
# dividing it by 12 leaves a figure still at or above the minimum. 

ANNUAL_MAX_THRESHOLD = 20500    # if higher than this, a maximum may be an annual figure
BOTH_ANNUAL_THRESHOLD = 40000   # a monthly minimum this high is rare apart from senior mgt roles

both = df_clean['salary_min_clean'].notna() & df_clean['salary_max_clean'].notna()
wide_range = both & (df_clean['salary_min_clean'] < 0.2 * df_clean['salary_max_clean'])
candidate_monthly = df_clean['salary_max_clean'] / 12

# BOTH masks are computed before either is applied. 
max_annual = (wide_range
              & (df_clean['salary_max_clean'] > ANNUAL_MAX_THRESHOLD)
              & (candidate_monthly >= df_clean['salary_min_clean']))

# A row that failed the test - too wide spread.
too_wide = wide_range & ~max_annual

# Maximum quoted annually, minimum monthly
df_clean.loc[max_annual, 'salary_max_clean'] = candidate_monthly[max_annual]

# If range too wide to - collapse both ends onto the raw average
df_clean.loc[too_wide, 'salary_min_clean'] = df_clean.loc[too_wide, 'average_salary']
df_clean.loc[too_wide, 'salary_max_clean'] = df_clean.loc[too_wide, 'average_salary']

# Both ends quoted annually. Evaluated after the steps above, so a minimum only
# revealed as annual by an earlier fix is still caught.
both_annual = df_clean['salary_min_clean'].notna() & \
              (df_clean['salary_min_clean'] > BOTH_ANNUAL_THRESHOLD)
for column in ['salary_min_clean', 'salary_max_clean']:
    df_clean.loc[both_annual, column] = df_clean.loc[both_annual, column] / 12

# Junior label on a very high monthly salary is a mismatch, not a conversion
junior_high = df_clean['positionLevels_clean'].isin(JUNIOR_LEVELS) & \
              (df_clean['salary_max_clean'] >= ANNUAL_MAX_THRESHOLD)
salary_level_mismatch = salary_level_mismatch | junior_high

assert not (max_annual & too_wide).any(), 'annual rules overlap - check the masks'

# None of the three rules can invert a range: the annual reading is refused
# unless it stays above the minimum, collapsing sets both ends equal, and
# dividing both ends preserves their order. This asserts that stays true.
inverted = both & (df_clean['salary_max_clean'] < df_clean['salary_min_clean'])
assert inverted.sum() == 0, (
    f'{inverted.sum():,} postings ended up with a maximum below their minimum')

print(f'Maximum converted from annual: {max_annual.sum():,}')
print(f'Range collapsed to average:    {too_wide.sum():,}')
print(f'Both ends from annual:         {both_annual.sum():,}')
print(f'Junior label, high pay:        {junior_high.sum():,} (flagged)')
print('No inverted ranges.')

Maximum converted from annual: 148
Range collapsed to average:    287
Both ends from annual:         302
Junior label, high pay:        46 (flagged)
No inverted ranges.


In [16]:
# Final clean

# converted average, and the mismatch flag the dashboard filters
df_clean['salary_min_clean'] = df_clean['salary_min_clean'].round()
df_clean['salary_max_clean'] = df_clean['salary_max_clean'].round()
df_clean['average_salary_clean'] = (
    df_clean[['salary_min_clean', 'salary_max_clean']].mean(axis=1)
)
df_clean['salary_level_mismatch'] = salary_level_mismatch

rescued = (df_clean['average_salary_clean'].between(SALARY_FLOOR, SALARY_CEILING)
           & ~df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING))
print(f'Postings rescued into the usable band by conversion: {rescued.sum():,}')
print()
print(df_clean[['average_salary', 'average_salary_clean']].describe().round(0))

Postings rescued into the usable band by conversion: 3,705

       average_salary  average_salary_clean
count        629246.0              627710.0
mean           5127.0                4966.0
std           32822.0                3292.0
min               1.0                   4.0
25%            2950.0                2975.0
50%            3950.0                4000.0
75%            6000.0                6000.0
max        12666400.0               65000.0


## 1.7 Parse Categories + id into list columns
We parse the categories into list columns on df_clean, then use .explode() to build the long table.

In [17]:
# Parse safely: return [] instead of raising, so one bad row cannot kill the run.

def parse_categories(json_string):
    if pd.isna(json_string):                                    # if empty, return [] instead of None
        return []
    try:
        items = json.loads(json_string)
    except (TypeError, ValueError):                             # if json.loads fail, return [] instead of Error
        return []
    if not isinstance(items, list):                             # if not list, return []
        return []
    # de-duplicate within a posting, keep original order
    seen, out = set(), []
    for c in items:
        if not isinstance(c, dict) or c.get('category') is None:
            continue
        name = str(c['category']).strip()
        if name in seen:
            continue
        seen.add(name)
        out.append((c.get('id'), name))
    return out


parsed = df_clean['categories'].apply(parse_categories)

# Two aligned list columns: same length per row, so they explode together later
df_clean['category_id_list'] = parsed.apply(lambda rows: [i for i, _ in rows])
df_clean['category_list']    = parsed.apply(lambda rows: [n for _, n in rows])
df_clean['n_categories']     = df_clean['category_list'].str.len()

# First category only for a quick groupby 
# User-facing filter must use category_list, not this.
df_clean['main_category'] = df_clean['category_list'].str[0]

print('Rows that failed to parse or were empty:', (df_clean['n_categories'] == 0).sum())
print()
print(df_clean['n_categories'].value_counts().sort_index())

Rows that failed to parse or were empty: 0

n_categories
1    417583
2    111329
3     52309
4     22422
5     25603
Name: count, dtype: int64


In [18]:
print(f"Postings in more than one industry: {(df_clean['n_categories'] > 1).mean():.1%}")
print(f"Postings with no industry at all:   {(df_clean['n_categories'] == 0).mean():.1%}")
print(f"Total postings:                     {len(df_clean):,}")
print(f"Total posting x category pairs:     {df_clean['n_categories'].sum():,}")

df_clean[['title', 'category_list', 'main_category', 'n_categories']].head(5)

Postings in more than one industry: 33.6%
Postings with no industry at all:   0.0%
Total postings:                     629,246
Total posting x category pairs:     1,014,871


,title,category_list,main_category,n_categories
15438,Quantity Surveyor - Structural Steel,"[Building and Construction, Engineering]",Building and Construction,2
20535,Preschool Teacher (Foreigner / Local),[Education and Training],Education and Training,1
23658,Haircut Specialist,"[Customer Service, Personal Care / Beauty, Sal...",Customer Service,4
23805,Senior Purchasing Executive (Marine),[Purchasing / Merchandising],Purchasing / Merchandising,1
17692,Assistant Chef,"[F&B, General Work]",F&B,2


## 1.8 .explode() into long table

In [19]:
df_categories_exploded = (
    df_clean[['metadata_jobPostId', 'category_id_list', 'category_list']]
    .explode(['category_id_list', 'category_list'])          # pass lists to explode, unnest in parallel
    .rename(columns={'category_id_list': 'category_id',
                     'category_list': 'category_name'})
    .dropna(subset=['category_name'])                        # drop the empty-category rows i.e. any rows with no categories
    .reset_index(drop=True)
)

# Ensuring same dtypes
df_categories_exploded['category_id'] = df_categories_exploded['category_id'].astype('int64')
df_categories_exploded['category_name'] = df_categories_exploded['category_name'].astype('category')

print(f"df_clean:                {len(df_clean):,} rows (one per posting)")
print(f"df_categories_exploded:  {len(df_categories_exploded):,} rows (one per posting x category)")
df_categories_exploded.head(8)

# Assign df_categories to exploded
df_categories = df_categories_exploded

df_clean:                629,246 rows (one per posting)
df_categories_exploded:  1,014,871 rows (one per posting x category)


## 1.8.1 Overview of all categories and ids

In [20]:
category_lookup = (
    df_categories[['category_id', 'category_name']]
    .drop_duplicates()
    .sort_values('category_id')
    .reset_index(drop=True)
)
print(f'{len(category_lookup)} distinct industries')

category_counts = df_categories['category_name'].value_counts()                     # count for each categories
category_lookup['postings'] = category_lookup['category_name'].map(category_counts) # map count using category name, to new column, postings
category_lookup

43 distinct industries


,category_id,category_name,postings
0,1,Accounting / Auditing / Taxation,49027
1,2,Admin / Secretarial,70357
2,3,Advertising / Media,11947
3,4,Architecture / Interior Design,9744
4,5,Banking and Finance,40026
5,6,Building and Construction,53918
6,7,Consulting,21389
7,8,Customer Service,62119
8,9,Design,12880
9,10,Education and Training,25157


## 1.9 Job Roles from 'title_clean'
Our brief covers benchmarking "by industry" and also "by role". 
This section covers the role dimension, using 'title_clean'

In [21]:
ROLE_PATTERNS = [
    ('Data / Analytics',           r'\b(data scientist|data analyst|data engineer|business intelligence|analytics|machine learning)\b'),
    ('Software Engineering',       r'\b(software engineer|developer|programmer|full stack|front end|back end|devops|qa engineer|test engineer)\b'),
    ('IT / Infrastructure',        r'\b(it (support|executive|manager|specialist)|system(s)? (admin|engineer|analyst)|network engineer|cyber ?security|cloud engineer|helpdesk|technical support)\b'),
    ('Product / Project',          r'\b(product manager|product owner|project manager|project executive|scrum master|business analyst)\b'),
    ('Design',                     r'\b(designer|creative director|art director|graphic)\b'),
    ('Engineering (Non-Software)', r'\b(mechanical|electrical|civil|structural|process|chemical|industrial|maintenance) engineer\b'),
    ('Sales / Business Dev',       r'\b(sales|business development|account (manager|executive)|relationship manager|retail assistant|promoter)\b'),
    ('Marketing / Comms',          r'\b(marketing|brand|content|social media|communications|public relations|copywriter)\b'),
    ('Finance / Accounting',       r'\b(account(s|ant|ing)|audit|tax|finance|financial|treasury|credit|bookkeep|payroll)\b'),
    ('Human Resources',            r'\b(hr|human resource|recruit|talent acquisition|people operations)\b'),
    ('Operations / Logistics',     r'\b(operations|logistics|warehouse|supply chain|procurement|purchasing|inventory|dispatch|driver|forklift)\b'),
    ('Healthcare',                 r'\b(nurse|nursing|doctor|physician|pharmacist|therapist|clinic|medical|dental|healthcare|caregiver)\b'),
    ('Education',                  r'\b(teacher|tutor|lecturer|trainer|educator|instructor|curriculum|childcare|preschool)\b'),
    ('Customer Service',           r'\b(customer service|customer support|call cent|service crew|receptionist|concierge|guest service)\b'),
    ('Food & Beverage',            r'\b(chef|cook|barista|waiter|waitress|kitchen|f b|restaurant|bartender|pastry)\b'),
    ('Legal / Compliance',         r'\b(legal|lawyer|solicitor|paralegal|compliance|regulatory|counsel)\b'),
    ('Admin / Secretarial',        r'\b(admin|administrative|secretar|clerk|clerical|data entry|personal assistant)\b'),
    ('Construction / Trades',      r'\b(construction|site (supervisor|engineer|manager)|foreman|carpenter|plumber|electrician|welder|painter|technician)\b'),
    ('Security / Facilities',      r'\b(security (officer|guard|supervisor)|cleaner|cleaning|housekeep|facilit|janitor)\b'),
]

ROLE_REGEX = [(label, re.compile(pattern)) for label, pattern in ROLE_PATTERNS]


def match_roles(title):                      # every role family the title matches
    if not isinstance(title, str) or not title:
        return []
    return [label for label, regex in ROLE_REGEX if regex.search(title)]


titles = df_clean['title_clean'].fillna('')
df_clean['role_list']    = titles.apply(match_roles)
df_clean['primary_role'] = df_clean['role_list'].apply(
    lambda r: r[0] if r else 'Other / Unclassified')

df_clean[['title', 'title_clean', 'primary_role', 'role_list']].head(10)

,title,title_clean,primary_role,role_list
15438,Quantity Surveyor - Structural Steel,quantity surveyor structural steel,Other / Unclassified,[]
20535,Preschool Teacher (Foreigner / Local),preschool teacher foreigner local,Education,[Education]
23658,Haircut Specialist,haircut specialist,Other / Unclassified,[]
23805,Senior Purchasing Executive (Marine),senior purchasing executive marine,Operations / Logistics,[Operations / Logistics]
17692,Assistant Chef,assistant chef,Food & Beverage,[Food & Beverage]
16957,Kitchen Crew,kitchen crew,Food & Beverage,[Food & Beverage]
18642,Assistant Chef,assistant chef,Food & Beverage,[Food & Beverage]
18694,Project Manager,project manager,Product / Project,[Product / Project]
16712,Service Crew,service crew,Customer Service,[Customer Service]
18762,Kitchen Crew,kitchen crew,Food & Beverage,[Food & Beverage]


## Analysis Grouping

In [22]:
# Salary usability

df_clean['salary_reliable'] = (
    df_clean['average_salary_clean'].between(SALARY_FLOOR, SALARY_CEILING)
    & ~df_clean['salary_level_mismatch']
)

print(f"Usable salary rows (converted): {df_clean['salary_reliable'].sum():,} "
      f"({df_clean['salary_reliable'].mean():.1%})")
print(f"df_clean_salary rows (raw):     {len(df_clean_salary):,}")
print(f"Net gain from conversion:       "
      f"{df_clean['salary_reliable'].sum() - len(df_clean_salary):+,}")

# The flag must be a superset of the raw filter, minus the level mismatches.
raw_ok = df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING)
lost = raw_ok & ~df_clean['salary_reliable'] & ~df_clean['salary_level_mismatch']
assert lost.sum() == 0, (
    f'{lost.sum():,} postings were usable before conversion and are not now - '
    'check the thresholds in 1.5.2')
print('\nNo posting was lost by the conversion.')

# Running describe on salary_reliable to confirm no outliers
print(df_clean.loc[df_clean['salary_reliable'],"average_salary_clean"].describe().round(0))

Usable salary rows (converted): 626,968 (99.6%)
df_clean_salary rows (raw):     623,309
Net gain from conversion:       +3,659

No posting was lost by the conversion.
count    626968.0
mean       4969.0
std        3283.0
min         500.0
25%        3000.0
50%        4000.0
75%        6000.0
max       55500.0
Name: average_salary_clean, dtype: float64


In [23]:
# Experience bands
def group_experience(years):
    if pd.isna(years):
        return None
    if years <= 1:
        return '0-1 years'
    if years <= 4:
        return '2-4 years'
    if years <= 9:
        return '5-9 years'
    return '10+ years'


# Employment type, grouped into standard vs flexible
EMPLOYMENT_GROUPS = {
    'Permanent': 'Standard', 'Full Time': 'Standard', 'Contract': 'Standard',
    'Part Time': 'Flexible / Non-Standard', 'Temporary': 'Flexible / Non-Standard',
    'Freelance': 'Flexible / Non-Standard', 'Flexi-work': 'Flexible / Non-Standard',
    'Internship/Attachment': 'Internship',
}

df_clean['experience_group'] = df_clean['minimumYearsExperience'].apply(group_experience)
df_clean['employment_group'] = df_clean['employmentTypes'].map(EMPLOYMENT_GROUPS).fillna('Other')
df_clean['posting_month']    = df_clean['metadata_originalPostingDate'].dt.to_period('M').dt.to_timestamp()

missing_exp = df_clean['experience_group'].isna().mean()
print(f'Postings with no experience value: {missing_exp:.1%} '
      '(excluded from any experience-band filter)\n')

for column in ['experience_group', 'employment_group']:
    print(f'--- {column} ---')
    print(df_clean[column].value_counts(dropna=False), end='\n\n')

Postings with no experience value: 0.0% (excluded from any experience-band filter)

--- experience_group ---
experience_group
2-4 years    264010
0-1 years    208858
5-9 years    128556
10+ years     27803
NaN              19
Name: count, dtype: int64

--- employment_group ---
employment_group
Standard                   598650
Flexible / Non-Standard     26472
Internship                   4124
Name: count, dtype: int64



## Writing Files
Output for 2 csv files
- SGJobData_clean.csv
- SGJobData_categories.csv : this is the exploded table for industry breakdown

In [62]:
SEP = '|'   # csv cannot store python lists, 3 list columns joined with |

df_out = df_clean.copy()
for column in ['category_list', 'category_id_list', 'role_list']:
    df_out[column] = df_out[column].apply(
        lambda items: SEP.join(str(i) for i in items) if len(items) else '')

df_out = df_out.drop(columns=['categories'], errors='ignore')

df_out.to_csv('SGJobData_clean.csv', index=False)
df_categories.to_csv('SGJobData_categories.csv', index=False)


## Streamlit Dashboard
Overview: Client picks a target industry or role and gets prevailing salary ranges and a read on hiring pool.

In [66]:
%%writefile app.py
"""
Singapore Jobs Dashboard
Talent cost + hiring pool benchmark for companies entering the Singapore market.

Run:  streamlit run app.py
Needs SGJobData_clean.csv and SGJobData_categories.csv (written by the notebook).


Counting rule
-------------
SGJobData_clean.csv is one row per job posting. 
Category filters .explode() to MATCH, then join the surviving job IDs 
back to the posting-level frame to COUNT. 

Without the join-back, a posting sitting in 3 industries counts 3 times
in every headline number. 

Charts that are *meant* to count per industry
(top categories, sunburst, scatter) use exploded_view().
"""

from pathlib import Path

import altair as alt
import pandas as pd
import plotly.express as px
import streamlit as st

HERE = Path(__file__).resolve().parent
CLEAN_PATH = HERE / "SGJobData_clean.csv"
CATEGORIES_PATH = HERE / "SGJobData_categories.csv"

SEP = "|"                
SALARY_FLOOR = 500       
SALARY_CEILING = 60_000   

DATE_COLS = ["metadata_expiryDate", "metadata_newPostingDate",
             "metadata_originalPostingDate"]

# Salary column produced by section 1.6.1 (hourly and annual converted to
# monthly). Falls back to the raw average if the notebook has not been re-run.
SALARY_COL = "average_salary_clean"


# 1. Page Configuration
st.set_page_config(
    page_title="SG Jobs Dashboard",
    page_icon="",
    layout="wide",
    initial_sidebar_state="expanded",
)

_W = ({"width": "stretch"}
      if tuple(int(p) for p in st.__version__.split(".")[:2]) >= (1, 49)
      else {"use_container_width": True})


# 2. Reading Clean Job Data (Cached for Performance)
def split_list_col(series):
    """CSV stores lists as 'A|B|C'. Turn them back into real lists."""
    return series.fillna("").apply(lambda s: [p for p in str(s).split(SEP) if p])


@st.cache_data(show_spinner="Loading postings...")
def load_data():
    df = pd.read_csv(CLEAN_PATH, parse_dates=DATE_COLS + ["posting_month"])
    df["category_list"] = split_list_col(df["category_list"])
    df["role_list"] = split_list_col(df["role_list"])
    if SALARY_COL not in df.columns:          # notebook not re-run yet
        df[SALARY_COL] = df["average_salary"]
    return df


@st.cache_data(show_spinner="Loading categories...")
def load_categories():
    return pd.read_csv(CATEGORIES_PATH)


@st.cache_data
def option_lists(_df, _cats):
    df, cats = _df, _cats
    return {
        "categories": sorted(cats["category_name"].dropna().unique().tolist()),
        "roles": sorted(df["role_list"].explode().dropna().unique().tolist()),
        "employment": sorted(df["employmentTypes"].dropna().unique().tolist()),
        "positions": sorted(df["positionLevels"].dropna().unique().tolist()),
        "experience": ["0-1 years", "2-4 years", "5-9 years", "10+ years"],
    }


df = load_data()
df_categories = load_categories()
opts = option_lists(df, df_categories)

# 3. Sidebar Filters
st.sidebar.header("Dashboard Filters")

if st.sidebar.button("Reset all filters", **_W):
    st.session_state.clear()
    st.rerun()

# Date filter. Uses originalPostingDate rather than newPostingDate so that a
# repost does not shift a job to a later month than it was first advertised.
date_col = "metadata_originalPostingDate"
min_date = df[date_col].min().to_pydatetime()
max_date = df[date_col].max().to_pydatetime()
selected_dates = st.sidebar.date_input(
    "Select Date Range", value=(min_date, max_date),
    min_value=min_date, max_value=max_date, key="f_date",
)

st.sidebar.subheader("Industry")
selected_categories = st.sidebar.multiselect(
    "Filter by Category", opts["categories"], key="f_cat",
    help="A posting can sit in several industries. 'Any' keeps postings in at "
         "least one; 'All' keeps only postings carrying every one selected.",
)
cat_mode = st.sidebar.radio(
    "Match", ["Any of these", "All of these"], horizontal=True,
    key="f_cat_mode", disabled=not selected_categories,
)

st.sidebar.subheader("Role")
selected_roles = st.sidebar.multiselect(
    "Filter by Role Family", opts["roles"], key="f_role",
    help="Derived from the job title in section 1.8 of the notebook.",
)
title_query = st.sidebar.text_input(
    "Title contains", key="f_title", placeholder="e.g. senior, analyst, nurse",
    help="Case-insensitive, searches title_clean. Comma-separate for OR.",
)

st.sidebar.subheader("Job Level & Type")
selected_pos = st.sidebar.multiselect(
    "Filter by Position Levels", opts["positions"], key="f_pos"
)
selected_type = st.sidebar.multiselect(
    "Filter by Employment Types", opts["employment"], key="f_emp"
)
selected_experience = st.sidebar.multiselect(
    "Filter by Experience Band", opts["experience"], key="f_exp"
)

st.sidebar.subheader("Salary")
salary_only = st.sidebar.checkbox(
    "Usable salary rows only", value=True, key="f_salok",
    help=f"Keeps {SALARY_COL} between S${SALARY_FLOOR:,} and S${SALARY_CEILING:,} "
         "after hourly and annual postings have been converted to monthly.",
)
min_n = st.sidebar.number_input(
    "Min postings for a benchmark", min_value=1, max_value=500, value=30, step=10,
    key="f_minn",
    help="Industries or roles with fewer postings are hidden from the median "
         "salary charts, since a median over a handful of postings is noise.",
)


# 4. Apply Filters to Dataset
#    Category and role explode to MATCH, then join back so that dff stays at
#    one row per posting.
def ids_matching_list_col(frame, column, wanted, mode="any"):
    long = (frame[["metadata_jobPostId", column]]
            .explode(column)
            .dropna(subset=[column]))
    hits = long[long[column].isin(wanted)]
    if mode == "all":
        counts = hits.groupby("metadata_jobPostId")[column].nunique()
        return counts[counts == len(wanted)].index
    return hits["metadata_jobPostId"].unique()


def apply_filters(df):
    dff = df

    if selected_categories:
        mode = "all" if cat_mode == "All of these" else "any"
        dff = dff[dff["metadata_jobPostId"].isin(
            ids_matching_list_col(dff, "category_list", selected_categories, mode))]

    if selected_roles:
        dff = dff[dff["metadata_jobPostId"].isin(
            ids_matching_list_col(dff, "role_list", selected_roles))]

    if title_query.strip():
        terms = [t.strip().lower() for t in title_query.split(",") if t.strip()]
        pattern = "|".join(pd.Series(terms).str.replace(
            r"([.^$*+?()\[\]{}|\\])", r"\\\1", regex=True))
        dff = dff[dff["title_clean"].str.contains(pattern, na=False, regex=True)]

    if selected_pos:
        dff = dff[dff["positionLevels"].isin(selected_pos)]
    if selected_type:
        dff = dff[dff["employmentTypes"].isin(selected_type)]
    if selected_experience:
        dff = dff[dff["experience_group"].isin(selected_experience)]

    if salary_only:
        dff = dff[dff["salary_reliable"]]

    # Date edge case: date_input returns a single date mid-selection
    if isinstance(selected_dates, tuple) and len(selected_dates) == 2:
        start, end = (pd.Timestamp(d) for d in selected_dates)
    else:
        start, end = pd.Timestamp(min_date), pd.Timestamp(max_date)
    dff = dff[dff[date_col].between(start, end + pd.Timedelta(days=1))]

    return dff


filtered_df = apply_filters(df)


def exploded_view(frame):
    """One row per posting x industry. For industry charts ONLY - a
    multi-industry posting appears once per industry, which is the intent
    here and wrong for any headline count."""
    long = (frame[["metadata_jobPostId", "title", "category_list", SALARY_COL,
                   "salary_reliable", "positionLevels", "employmentTypes",
                   "numberOfVacancies"]]
            .explode("category_list")
            .dropna(subset=["category_list"])
            .rename(columns={"category_list": "category"}))
    long["category"] = long["category"].astype(str)
    return long


# 5. Dashboard Header
st.title("Singapore Jobs Dashboard")
st.markdown(
    "Prevailing salary ranges and hiring pool depth by industry and role. "
    "All salaries are monthly SGD, with hourly and annual postings converted."
)

if filtered_df.empty:
    st.warning("No data available for the selected filters.")
    st.stop()

st.caption(
    f"Showing **{len(filtered_df):,}** of {len(df):,} postings "
    f"({len(filtered_df) / len(df):.1%}) · "
    f"{filtered_df[date_col].min():%b %Y} – {filtered_df[date_col].max():%b %Y}"
)
st.divider()


# 6. Top Row Layout: Metric Cards
#    Deltas compare this selection against the whole market, computed live.
salary_rows = filtered_df[filtered_df["salary_reliable"]]
market = df[df["salary_reliable"]]

median_salary = salary_rows[SALARY_COL].median() if len(salary_rows) else float("nan")
mean_salary = salary_rows[SALARY_COL].mean() if len(salary_rows) else float("nan")
market_median = market[SALARY_COL].median()

monthly_vacancies = (
    filtered_df.groupby(pd.Grouper(key=date_col, freq="M"))["numberOfVacancies"]
    .sum().mean()
)

col1, col2, col3, col4 = st.columns(4)
col1.metric(
    label="Median Salary",
    value=f"${median_salary:,.0f}" if pd.notna(median_salary) else "N/A",
    delta=f"{median_salary - market_median:+,.0f} vs market"
    if pd.notna(median_salary) else None,
)
col2.metric(
    label="Mean Salary",
    value=f"${mean_salary:,.0f}" if pd.notna(mean_salary) else "N/A",
    delta=f"{mean_salary - median_salary:+,.0f} vs median"
    if pd.notna(mean_salary) else None,
    help="Mean above median means a tail of high-paying postings pulling it up.",
)
col3.metric(label="Hiring Companies", value=f"{filtered_df['postedCompany_name'].nunique():,}")
col4.metric(
    label="Avg Monthly Vacancies",
    value=f"{monthly_vacancies:,.0f}" if pd.notna(monthly_vacancies) else "N/A",
)

if salary_only and len(filtered_df) != len(salary_rows):
    st.caption(
        f"{len(filtered_df) - len(salary_rows):,} postings in this selection have "
        "no usable salary and are excluded from the salary figures above."
    )

st.divider()


# 7. Middle Row Layout: Core Analytics Charts
chart_col1, chart_col2 = st.columns(2)

with chart_col1:
    st.subheader("Number of Vacancies Over Time")
    trend_df = (filtered_df.groupby("posting_month")["numberOfVacancies"]
                .sum().reset_index())
    fig_trend = px.line(
        trend_df, x="posting_month", y="numberOfVacancies",
        template="plotly_white",
        labels={"numberOfVacancies": "Vacancies", "posting_month": "Posting Month"},
    )
    fig_trend.update_layout(margin=dict(l=20, r=20, t=10, b=20))
    st.plotly_chart(fig_trend, **_W)

with chart_col2:
    # Changed from a pie chart: pie slices imply parts of a whole, and medians
    # do not sum to a whole. A bar chart compares levels correctly.
    st.subheader("Median Salary per Position Level")
    if salary_rows.empty:
        st.info("No usable salary rows in this selection.")
    else:
        pos_df = (salary_rows.groupby("positionLevels")[SALARY_COL]
                  .agg(median="median", postings="size").reset_index())
        pos_df = pos_df[pos_df["postings"] >= min_n].sort_values("median", ascending=False)
        if pos_df.empty:
            st.info(f"No position level has at least {min_n} usable-salary postings.")
        else:
            pos_df["positionLevels"] = pos_df["positionLevels"].astype(str)
            fig_pos = px.bar(
                pos_df, x="median", y="positionLevels", orientation="h",
                text_auto=".3s", color="median",
                color_continuous_scale=px.colors.sequential.RdBu,
                labels={"median": "Median Monthly Salary (S$)",
                        "positionLevels": "Position Level"},
            )
            fig_pos.update_layout(margin=dict(l=20, r=20, t=10, b=20),
                                  yaxis={"categoryorder": "total ascending"},
                                  coloraxis_showscale=False)
            st.plotly_chart(fig_pos, **_W)

st.divider()


# 8. Bottom Row Layout: Categorical Performance & Heatmap
lower_col1, lower_col2 = st.columns([3, 2])

long_all = exploded_view(filtered_df)
long_salary = long_all[long_all["salary_reliable"]]

with lower_col1:
    st.subheader("Median Salary, Top 10 Industries")
    st.caption(
        "Exploded by industry, so a posting in several industries is counted "
        "in each. Industries with fewer than "
        f"{min_n} usable-salary postings are omitted."
    )
    if long_salary.empty:
        st.info("No usable salary rows in this selection.")
    else:
        cat_df = (long_salary.groupby("category")[SALARY_COL]
                  .agg(median="median", postings="size").reset_index())
        cat_df = (cat_df[cat_df["postings"] >= min_n]
                  .sort_values("median", ascending=False).head(10))
        if cat_df.empty:
            st.info(f"No industry has at least {min_n} usable-salary postings.")
        else:
            fig_bar = px.bar(
                cat_df, x="median", y="category", orientation="h",
                text_auto=".3s", color="median",
                color_continuous_scale=px.colors.sequential.Viridis,
                hover_data={"postings": True},
                labels={"median": "Median Monthly Salary (S$)", "category": ""},
            )
            fig_bar.update_layout(margin=dict(l=20, r=20, t=10, b=20),
                                  yaxis={"categoryorder": "total ascending"},
                                  coloraxis_showscale=False)
            st.plotly_chart(fig_bar, **_W)

with lower_col2:
    st.subheader("Position x Employment Salary Heatmap")
    st.caption("Median salary per cell. Blank cells have no postings.")

    def make_heatmap(input_df, input_y, input_x, input_color, input_color_theme):
        return (
            alt.Chart(input_df).mark_rect().encode(
                y=alt.Y(f"{input_y}:O", axis=alt.Axis(
                    title="", titleFontSize=18, titlePadding=15,
                    titleFontWeight=900, labelAngle=0)),
                x=alt.X(f"{input_x}:O", axis=alt.Axis(
                    title="", titleFontSize=18, titlePadding=15,
                    titleFontWeight=900, labelAngle=-45)),
                # median, not max: one outlier posting should not light up a cell
                color=alt.Color(f"median({input_color}):Q", legend=None,
                                scale=alt.Scale(scheme=input_color_theme)),
                tooltip=[alt.Tooltip(f"{input_y}:O"), alt.Tooltip(f"{input_x}:O"),
                         alt.Tooltip(f"median({input_color}):Q",
                                     title="Median salary", format=",.0f"),
                         alt.Tooltip("count():Q", title="Postings")],
                stroke=alt.value("black"),
                strokeWidth=alt.value(0.25),
            ).properties(height=380)
        )

    if salary_rows.empty:
        st.info("No usable salary rows in this selection.")
    else:
        heat_src = salary_rows[["positionLevels", "employmentTypes", SALARY_COL]].dropna()
        heat_src = heat_src.astype({"positionLevels": str, "employmentTypes": str})
        st.altair_chart(
            make_heatmap(heat_src, "positionLevels", "employmentTypes",
                         SALARY_COL, "blues"),
            **_W,
        )

st.divider()


# 9. Distribution & Hierarchy
chart_col3, chart_col4 = st.columns(2)

with chart_col3:
    st.subheader("Salary Spread by Industry")
    st.caption("Each point is a posting. Sampled to 5,000 for responsiveness.")
    if long_salary.empty:
        st.info("No usable salary rows in this selection.")
    else:
        sample = long_salary.sample(min(len(long_salary), 5_000), random_state=0)
        sample = sample.astype({"category": str, "positionLevels": str})
        chart = alt.Chart(sample).mark_circle(opacity=0.4).encode(
            x=alt.X("category:N", title=None, axis=alt.Axis(labelAngle=-45)),
            y=alt.Y(f"{SALARY_COL}:Q", title="Monthly Salary (S$)"),
            color=alt.Color("positionLevels:N", title="Position Level"),
            tooltip=["title:N", "category:N", "positionLevels:N",
                     alt.Tooltip(f"{SALARY_COL}:Q", format=",.0f")],
        ).properties(height=420).interactive()
        st.altair_chart(chart, theme="streamlit", **_W)

with chart_col4:
    st.subheader("Industry to Position to Employment Type")
    st.caption("Ring sizes count posting x industry pairs, not unique postings.")
    sun_src = long_all[["category", "positionLevels", "employmentTypes"]].dropna()
    if sun_src.empty:
        st.info("Not enough data for the sunburst in this selection.")
    else:
        sun_src = sun_src.astype(str)
        fig_sun = px.sunburst(
            sun_src, path=["category", "positionLevels", "employmentTypes"],
            color_discrete_sequence=px.colors.qualitative.Pastel,
        )
        fig_sun.update_layout(margin=dict(l=10, r=10, t=5, b=10), height=420)
        st.plotly_chart(fig_sun, **_W)

st.divider()


# 10. Raw Data
with st.expander("View filtered postings"):
    show_cols = ["title", "postedCompany_name", "main_category", "primary_role",
                 "positionLevels", "employmentTypes", "minimumYearsExperience",
                 "experience_group", "salary_min_clean", "salary_max_clean",
                 SALARY_COL, "status_jobStatus", date_col]
    show_cols = [c for c in show_cols if c in filtered_df.columns]
    st.dataframe(filtered_df[show_cols].head(1000), hide_index=True, **_W)
    st.caption("First 1,000 rows shown.")
    st.download_button(
        "Download filtered rows (CSV)",
        filtered_df[show_cols].to_csv(index=False).encode("utf-8"),
        file_name="filtered_jobs.csv", mime="text/csv",
    )


Overwriting app.py
